In [2]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\test'

Test du fichier prediction.py

In [3]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

In [74]:
import joblib 
import pandas as pd
from pathlib import Path
from mlProject.constants import *
from mlProject.utils.common import read_yaml

In [25]:
df_transf_data = pd.read_csv(Path("artifacts/data_transformation/transf_data_file.csv"))
df_transf_data

,Unnamed: 0,mean_input_data,max_input_data,min_input_data
0,area,5150.541284,16200,1650
1,bedrooms,2.965138,6,1
2,bathrooms,1.286239,4,1
3,stories,1.805505,4,1
4,mainroad,0.858716,1,0
5,guestroom,0.177982,1,0
6,basement,0.350459,1,0
7,hotwaterheating,0.045872,1,0
8,airconditioning,0.315596,1,0
9,parking,0.693578,3,0


In [26]:
#  input_data = [area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus]

input_data = [7420,4,2,3,"yes","no","no","no","yes",2,"yes","furnished"]

In [75]:
schema = read_yaml(SCHEMA_FILE_PATH).COLUMNS

[2026-06-26 18:53:53,383: INFO: common: yaml file: yaml file\schema.yaml loaded successfully]


In [76]:
column_str_name = [k for k, v in schema.items() if v == "str"]

In [77]:
column_str_name

['mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'prefarea',
 'furnishingstatus']

In [28]:
encoder = joblib.load(Path("artifacts/data_transformation/encoder.pkl"))

In [ ]:
str_values = [x for x in input_data if isinstance(x, str)]

encoded_str_values = []

for value, col in zip(str_values, column_str_name):
    encoded_value = encoder[col].transform([value])[0]
    encoded_str_values.append(encoded_value)

print(encoded_str_values)

[1, 0, 0, 0, 1, 1, 0]


In [31]:
# remplaer str dans input data par [1, 0, 0, 0, 1, 1, 0]

print("Original:", input_data)

# Récupérer les indices des valeurs string
string_indices = [i for i, value in enumerate(input_data) if isinstance(value, str)]

# Remplacer chaque valeur string par la valeur correspondante du mapping
for idx, map_val in zip(string_indices, encoded_str_values):
    input_data[idx] = map_val

print("Encodé:  ", input_data)

Original: [7420, 4, 2, 3, 'yes', 'no', 'no', 'no', 'yes', 2, 'yes', 'furnished']
Encodé:   [7420, 4, 2, 3, 1, 0, 0, 0, 1, 2, 1, 0]


In [32]:
input_normalized_data = (input_data - df_transf_data ["mean_input_data"]) / (df_transf_data ["max_input_data"] - df_transf_data ["min_input_data"])

In [33]:
input_normalized_data

0     0.155977
1     0.206972
2     0.237920
3     0.398165
4     0.141284
5    -0.177982
6    -0.350459
7    -0.045872
8     0.684404
9     0.435474
10    0.765138
11   -0.534862
dtype: float64

In [36]:
import numpy as np

In [ ]:
# predict new data
model = joblib.load(Path('artifacts/model_trainer/model.joblib'))

In [41]:
model.predict(np.array(input_normalized_data).reshape(1, 12))

array([6865222.0876264])

Update the entity

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PredictionNewDataConfig:
    model_path: Path
    transf_data_file: Path
    encoder_file: Path
    column_str_name: str

Update the configuration manager

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_prediction_new_data_config(self) -> PredictionNewDataConfig:
        config = self.config.prediction_new_data
        schema = self.schema.COLUMNS

        prediction_new_data_config = PredictionNewDataConfig(
            model_path = config.model_path,
            transf_data_file = config.transf_data_file,
            encoder_file = config.encoder_file,
            column_str_name = [k for k, v in schema.items() if v == "str"]
        )

        return prediction_new_data_config

Update the components

In [8]:
import joblib 
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
 #column_name = ["mainroad", "guestroom", "basement", "hotwaterheating",
              # "airconditioning", "prefarea", "furnishingstatus"]

In [12]:
# Ajouter méthode de noramlisation, PCA etc si vous avez en training
class PredictionNewDataPipeline:
    def __init__(self, config: PredictionNewDataConfig):
        self.config = config

        
    def Normalize_input_data(self, input_data):
        
        df_transf_data = pd.read_csv(self.config.transf_data_file)
        
        encoder =  joblib.load(self.config.encoder_file)
        
        str_values = [x for x in input_data if isinstance(x, str)]

        encoded_str_values = []

        for value, col in zip(str_values, self.config.column_str_name):
            encoded_value = encoder[col].transform([value])[0]
            encoded_str_values.append(encoded_value)

        # remplaer str dans input data par [1, 0, 0, 0, 1, 1, 0]
        # Récupérer les indices des valeurs string
        string_indices = [i for i, value in enumerate(input_data) if isinstance(value, str)]

        # Remplacer chaque valeur string par la valeur correspondante du mapping
        for idx, map_val in zip(string_indices, encoded_str_values):
            input_data[idx] = map_val

        input_normalized_data = (input_data - df_transf_data ["mean_input_data"]) / (df_transf_data ["max_input_data"] - df_transf_data ["min_input_data"])
        
        return input_normalized_data
    
    def predict(self, input_normalized_data):
        model = joblib.load(self.config.model_path)
        prediction = model.predict(np.array(input_normalized_data).reshape(1, 12))

        print(type(model))
        print(model)

        return prediction

Update the pipline

In [14]:
input_data_1 = [7420,4,2,3,"yes","no","no","no","yes",2,"yes","furnished"]

try:
 config = ConfigurationManager()   
 pred_new_data_config = config.get_prediction_new_data_config()
 pred_new_data = PredictionNewDataPipeline (config = pred_new_data_config)
 data_normalized_1 = pred_new_data.Normalize_input_data(input_data_1)
 pred_new_data.predict(data_normalized_1)
except Exception as e:
    raise e

[2026-06-26 19:35:33,640: INFO: common: yaml file: yaml file\config.yaml loaded successfully]
[2026-06-26 19:35:33,676: INFO: common: yaml file: yaml file\params.yaml loaded successfully]
[2026-06-26 19:35:33,676: INFO: common: yaml file: yaml file\schema.yaml loaded successfully]
[2026-06-26 19:35:33,701: INFO: common: created directory at: artifacts]
<class 'sklearn.linear_model._coordinate_descent.ElasticNet'>
ElasticNet(alpha=0.05, l1_ratio=0.99, random_state=42)


In [ ]:
print(pred_new_data.predict(data_normalized_1))

<class 'sklearn.linear_model._coordinate_descent.ElasticNet'>
ElasticNet(alpha=0.05, l1_ratio=0.99, random_state=42)
7990589.004525876
